# Setting up Python with `uv` inside Docker

For this lecture, we use a Docker container as the base system and then manage Python with `uv` inside that container. This avoids most host-specific dependency issues and gives us reproducible environments for notebooks and scripts.

You already have a conda-based setup notebook. This notebook focuses on the Docker + `uv` workflow, which is often faster for package resolution and cleaner for project-local environments.

## Why this stack?

- Docker gives us a stable Linux base image with R/Bioconductor and system libraries.
- `uv` gives us fast Python dependency management and project-local virtual environments.
- We can mix R-native workflows, `reticulate`, and Python in one reproducible setup.

## 1. Full Dockerfile (copy/paste)

Below is one complete Dockerfile that combines the Bioconductor Image with extra system dependencies (including fonts) used for `rpy2` + `uv` workflows.

Depending on different installation paths, we may or may not manually export some settings inside the docker container:

```
export R_HOME="$(R RHOME)"
export CC=/usr/bin/gcc
export CXX=/usr/bin/g++
export LDFLAGS="-L/usr/local/lib/R/lib -L/usr/lib/x86_64-linux-gnu"
export CPPFLAGS="-I/usr/include -I/usr/include/tirpc"
export LD_LIBRARY_PATH="/usr/local/lib/R/lib:/usr/lib/x86_64-linux-gnu:${LD_LIBRARY_PATH}"
```

In [ ]:
#| eval: false
# Base image: Bioconductor (R 4.6 + RStudio)
FROM bioconductor/bioconductor_docker:RELEASE_3_23

ENV LANG=C.UTF-8
ENV LC_ALL=C.UTF-8

# Environment variables
ENV CONDA_DIR=/opt/conda
ENV PATH=${CONDA_DIR}/bin:/root/.cargo/bin:$PATH
ENV MAKE="make"
ENV MAKEFLAGS="-j4"

# System dependencies (combined original + extra rpy2/plotting deps)
RUN apt-get update && apt-get install -y --no-install-recommends \
    curl \
    git \
    build-essential \
    gcc \
    g++ \
    gfortran \
    make \
    cmake \
    pkg-config \
    libpcre2-dev \
    libdeflate-dev \
    liblzma-dev \
    zlib1g-dev \
    libbz2-dev \
    libzstd-dev \
    libicu-dev \
    libtirpc-dev \
    libxml2-dev \
    libssl-dev \
    libcurl4-openssl-dev \
    libv8-dev \
    libhdf5-dev \
    libffi-dev \
    libreadline-dev \
    libsqlite3-dev \
    tk-dev \
    libpng-dev \
    libjpeg-dev \
    libfreetype6-dev \
    fontconfig \
    fonts-dejavu-core \
    fonts-dejavu-extra \
    fonts-liberation \
    fonts-freefont-ttf \
    && apt-get clean && rm -rf /var/lib/apt/lists/*

# Refresh font cache
RUN fc-cache -fv

# Switch user so Rust installs into /home/rstudio/.cargo
USER rstudio

# Install Rust
RUN curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y

# Make cargo available in shell
ENV PATH="/home/rstudio/.cargo/bin:${PATH}"

# Make cargo available to R sessions (.Renviron is read by R)
RUN echo 'PATH=${PATH}:/home/rstudio/.cargo/bin' >> /home/rstudio/.Renviron

# Switch back to root
USER root

# Install Miniforge3
RUN curl -L https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -o miniforge.sh \
    && bash miniforge.sh -b -p ${CONDA_DIR} \
    && rm miniforge.sh \
    && conda config --set auto_update_conda False

# Install uv from official image
COPY --from=ghcr.io/astral-sh/uv:latest /uv /uvx /usr/bin/

# Configure R + reticulate to use uv-managed environments
ENV RETICULATE_PYTHON_PREFERENCE=system
ENV RETICULATE_USE_MANAGED_VENV=yes

# Pre-install core R packages
RUN R -e "BiocManager::install(c('scran', 'scater', 'muscat', 'Rhdf5lib', 'edgeR', 'limma', 'glmGamPoi', 'DESeq2'), ask=FALSE)"
RUN R -e "BiocManager::install(c('basilisk', 'SingleCellExperiment', 'SparseArray', 'scrapper', 'sketchR'), ask=FALSE)"

RUN R -e "BiocManager::install(c('alabaster.base', 'alabaster.sce'), ask=FALSE)"

RUN R -e "BiocManager::install(c('lemur', 'MOFA2'), ask=FALSE)"
RUN R -e "BiocManager::install(c('decoupleR', 'clusterProfiler'), ask=FALSE)"
RUN R -e "BiocManager::install(c('ggplot2', 'scattermore', 'patchwork', 'cowplot', 'ComplexHeatmap', 'pheatmap', 'ggrastr'), ask=FALSE)"
RUN R -e "BiocManager::install(c('ggsci'), ask=FALSE)"
RUN R -e "BiocManager::install(c('reticulate', 'anndataR'), ask=FALSE)"

RUN R -e "BiocManager::install(c('tidyverse'), ask=FALSE)"


# Required installs to use R in Jupyter notebooks:

# Update and install the jupyter-client binary
RUN apt-get update
RUN apt-get install -y jupyter-client


RUN R -e "install.packages('IRkernel')"
RUN R -e "IRkernel::installspec(user = FALSE)"


RUN R -e "BiocManager::install(c('airway', 'TENxPBMCData'))"

WORKDIR /home/rstudio/project

## 2. Build the Docker image

Assuming your Dockerfile is in the current directory, build the image with a lecture tag:

In [ ]:
#| eval: false
docker build -t sc-lecture:bioc-r45-uv .

This image contains:

- Bioconductor base image (R 4.5, RStudio)
- Rust toolchain (for packages requiring Rust)
- Miniforge/conda tooling
- `uv` binary
- selected R single-cell and interoperability packages

## 3. Start a container

Mount your project directory and expose useful ports (RStudio and Jupyter):

In [ ]:
#| eval: false
docker run --rm -it \
  --name sc-lecture \
  -p 8787:8787 \
  -p 8888:8888 \
  -v "$(pwd)":/home/rstudio/project \
  sc-lecture:bioc-r45-uv

If you start it detached, you can enter it later using `docker exec`.

In [ ]:
#| eval: false
docker run -d --rm \
  --name sc-lecture \
  -p 8787:8787 -p 8888:8888 \
  -v "$(pwd)":/home/rstudio/project \
  sc-lecture:bioc-r45-uv

docker exec -it sc-lecture bash

## 4. Verify tools in container

Inside the container, check that `uv`, Python, and R are available:

In [ ]:
#| eval: false
uv --version
python --version
R --version

## 5. Install extra OS packages (if needed)

Some Python or R packages require system libraries at compile/runtime. If your base image does not already include what you need:

In [ ]:
#| eval: false
apt-get update
apt-get install -y --no-install-recommends \
  build-essential gcc g++ gfortran make cmake pkg-config \
  libpcre2-dev libdeflate-dev liblzma-dev zlib1g-dev libbz2-dev libzstd-dev \
  libicu-dev libtirpc-dev libxml2-dev libssl-dev libcurl4-openssl-dev \
  libpng-dev libjpeg-dev libfreetype6-dev libhdf5-dev

apt-get install -y --no-install-recommends \
  fontconfig fonts-dejavu-core fonts-dejavu-extra fonts-liberation fonts-freefont-ttf

fc-cache -fv

In your current Dockerfile, most critical dependencies are already installed, so this step is often unnecessary during day-to-day work.

## 6. `uv` project strategy for data analysis

`uv` was designed for software projects, but in analysis workflows we often want different dependency sets per stage. For example, preprocessing, integration, and VDJ analysis can each have their own environment.

In [ ]:
#| eval: false
project/
  01_preprocessing/
    pyproject.toml
    uv.lock
    .venv/
    preprocess.ipynb
    run_preprocess.py

  02_totalvi/
    pyproject.toml
    uv.lock
    .venv/
    totalvi.ipynb
    run_totalvi.py

  03_vdj/
    pyproject.toml
    uv.lock
    .venv/
    clonotypes.ipynb

This pattern improves reproducibility and avoids one oversized environment for everything.

## 6. Create one stage-specific `uv` project

Example for `01_preprocessing`:

In [ ]:
#| eval: false
cd /home/rstudio/project/project
mkdir -p 01_preprocessing
cd 01_preprocessing

uv init --python 3.12
uv add scanpy anndata pandas numpy matplotlib jupyterlab ipykernel
uv sync

If you already wrote a `pyproject.toml`, install exactly what is specified there with:

In [ ]:
#| eval: false
uv sync

## 7. Register a Jupyter kernel from this `uv` environment

This lets notebooks in JupyterLab use the environment of this stage:

In [ ]:
#| eval: false
uv run python -m ipykernel install \
  --user \
  --name sc-preprocess-uv \
  --display-name "Python (sc preprocess uv)"

Start JupyterLab inside that stage folder:

In [ ]:
#| eval: false
uv run jupyter lab --ip=0.0.0.0 --port=8888 --no-browser

## 8. Repeat per analysis stage

Create one `uv` project per stage and install only what that stage needs.

In [ ]:
#| eval: false
cd /home/rstudio/project/project/02_totalvi
uv init --python 3.12
uv add scvi-tools anndata pandas numpy jupyterlab ipykernel
uv sync

cd /home/rstudio/project/project/03_vdj
uv init --python 3.12
uv add scirpy awkward pandas numpy jupyterlab ipykernel
uv sync

## 9. Run scripts with project environments

Because each stage has its own `.venv`, running scripts stays isolated and reproducible:

In [ ]:
#| eval: false
cd /home/rstudio/project/project/01_preprocessing
uv run python run_preprocess.py

cd /home/rstudio/project/project/02_totalvi
uv run python run_totalvi.py

## 10. One-off script dependencies with uv

For quick utility scripts, `uv` can create a temporary environment from inline dependency metadata. This is convenient for scripts that are not part of a full project.

In [ ]:
#| eval: false
# Example script header (PEP 723 style):
# /// script
# dependencies = [
#   "pandas",
#   "anndata"
# ]
# ///

uv run quick_check.py

## 11. Practical notes

- Commit `pyproject.toml` and `uv.lock` for each stage.
- Do not commit `.venv/`.
- Keep environment scopes narrow; add packages only where needed.
- If plotting looks wrong in headless/container environments, ensure fonts are installed.
- If a package build fails, check missing OS libraries first (not only Python versions).

## 12. Troubleshooting examples

When package combinations conflict or binary wheels are unavailable, these commands are commonly useful:

In [ ]:
#| eval: false
uv pip install --force-reinstall --upgrade matplotlib
uv pip install --pre --upgrade "plotnine>=0.16.0a0"

This keeps the lecture workflow modular: Docker controls the system foundation, and `uv` controls Python dependencies per analysis stage.

Please note, 